<a href="https://colab.research.google.com/github/Moatemri/machine-learning-zoomcamp/blob/main/03-classification/homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

In [4]:
!wget wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


--2026-09-28 13:27:40--  http://wget/
Resolving wget (wget)... failed: Name or service not known.
wget: unable to resolve host address ‘wget’
--2026-09-28 13:27:40--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.03s   

2026-09-28 13:27:40 (8.35 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]

FINISHED --2026-09-28 13:27:40--
Total wall clock time: 0.3s
Downloaded: 1 files, 272K in 0.03s (8.35 MB/s)


In [5]:
df = pd.read_csv("course_lead_scoring_2026.csv")

In [6]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [7]:
df.nunique()

,0
lead_source,5
industry,6
employment_status,4
location,5
annual_income,4313
number_of_courses_viewed,7
interaction_count,14
lead_score,99
converted,2


In [8]:
df.dtypes

,0
lead_source,object
industry,object
employment_status,object
location,object
annual_income,float64
number_of_courses_viewed,int64
interaction_count,int64
lead_score,float64
converted,int64


In [9]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [10]:
categorical_columns =list(df.dtypes[df.dtypes == 'str'].index)
for c in categorical_columns:
        df[c] =df[c].str.lower().str.replace(' ','_')

In [11]:
numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

In [12]:
categorical = [
    col for col in df.columns
if col not in numerical and col != 'converted'
]

In [13]:
df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [14]:
df[numerical]=df[numerical].fillna(0)

In [15]:
df[categorical]= df[categorical].fillna('NA')

# Question 1

In [16]:
df.industry.value_counts()

,count
industry,
technology,1173
retail,925
healthcare,840
finance,720
education,639
manufacturing,463
NA,240


In [17]:
df.industry.mode()

,industry
0,technology


# Question 2

In [18]:
df.interaction_count.corr(df.lead_score)

np.float64(0.9157457980418677)

In [19]:
df.number_of_courses_viewed.corr(df.lead_score)

np.float64(0.7572042392104887)

In [20]:
df.number_of_courses_viewed.corr(df.interaction_count)

np.float64(0.7216090038937009)

In [21]:
df.annual_income.corr(df.interaction_count)

np.float64(0.12284235135955954)

In [22]:
print(abs(df.interaction_count.corr(df.lead_score)))
print(abs(df.number_of_courses_viewed.corr(df.lead_score)))
print(abs(df.number_of_courses_viewed.corr(df.interaction_count)))
print(abs(df.annual_income.corr(df.interaction_count)))


0.9157457980418677
0.7572042392104887
0.7216090038937009
0.12284235135955954


In [23]:
# Question 3

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [26]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [27]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [28]:
from sklearn.metrics import mutual_info_score

In [29]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

In [30]:
mi = df_train[categorical].apply(mutual_info_converted_score)
score =mi.sort_values(ascending=False)
round(score,2)

,0
lead_source,0.03
employment_status,0.02
industry,0.00
location,0.00


# Question 4

In [98]:
from sklearn.feature_extraction import DictVectorizer

In [99]:
dv = DictVectorizer(sparse=False)

In [100]:
train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [101]:
from sklearn.linear_model import LogisticRegression

In [102]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [103]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [104]:
y_pred = model.predict_proba(X_val)[:, 1]

In [105]:
convert_decision = (y_pred >= 0.5)

In [108]:
accuracy_original =(y_val == convert_decision).mean()
#round(accuracy, 2)
accuracy

np.float64(0.645)

In [40]:
# Question 5

In [116]:
features =[categorical + numerical]
features




[['lead_source',
  'industry',
  'employment_status',
  'location',
  'annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score']]

In [112]:
for feature in features[0]:
    features_subset = [f for f in features[0] if f != feature]
    #display(features_subset)
    dv = DictVectorizer(sparse=False)
    train_dict = df_train[features_subset].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[features_subset].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    convert_decision = (y_pred >= 0.5)
    accuracy =(y_val == convert_decision).mean()
    #print("Ohne:", feature, "accurray is", accuracy)
    print ()

    print("Ohne", feature,"    differnce accuracy is ",accuracy_original - accuracy )


Ohne lead_source     differnce accuracy is  0.0030000000000000027

Ohne industry     differnce accuracy is  0.0

Ohne employment_status     differnce accuracy is  0.0

Ohne location     differnce accuracy is  0.0

Ohne annual_income     differnce accuracy is  -0.07899999999999996

Ohne number_of_courses_viewed     differnce accuracy is  0.0020000000000000018

Ohne interaction_count     differnce accuracy is  0.04400000000000004

Ohne lead_score     differnce accuracy is  0.0010000000000000009


In [43]:
# question 6

In [120]:
for varr in [0.000001, 0.00001, 0.0001, 0.001]:

    dv = DictVectorizer(sparse=False)
    train_dict = df_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[features_subset].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    model = LogisticRegression(solver='liblinear', C=varr, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    convert_decision = (y_pred >= 0.5)
    accuracy =(y_val == convert_decision).mean()
    round(accuracy,3)
    print("for C =", varr,"     accuracy is ",accuracy )

for C = 1e-06      accuracy is  0.598
for C = 1e-05      accuracy is  0.598
for C = 0.0001      accuracy is  0.613
for C = 0.001      accuracy is  0.647
